# Unsupervised Training with SimCSE

**Purpose:** Compare unsupervised domain adaptation vs curriculum contrastive learning.

**Method:** SimCSE (Simple Contrastive Learning of Sentence Embeddings)
- Uses dropout as augmentation - same sentence through model twice = different embeddings
- No labeled pairs needed - uses raw ticket descriptions
- Works with any encoder model (including MPNet)

**Why not TSDAE?** MPNet is encoder-only and doesn't support being used as a decoder.

**Reference:** [SimCSE Paper](https://arxiv.org/abs/2104.08821)

In [8]:
# Cell 1: Imports and Setup
import os
import sys
import json
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import roc_auc_score

# Check device
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using MPS (Apple Silicon)")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"PyTorch version: {torch.__version__}")

Using CUDA: NVIDIA GeForce RTX 5090
PyTorch version: 2.9.1+cu130


In [ ]:
# Cell 2: Configuration
CONFIG = {
    # Model
    'base_model': 'sentence-transformers/all-mpnet-base-v2',
    
    # SimCSE hyperparameters
    'batch_size': 64,       # Larger batch = more negatives = better
    'epochs': 1,            # SimCSE converges quickly
    'learning_rate': 3e-5,
    'warmup_ratio': 0.1,
    
    # Data
    'data_path': 'data/incident_ticket_data_redacted.csv',
    'text_column': 'cleaned_text',  # Use cleaned text
    'max_samples': None,  # None = use all
    
    # Evaluation
    'test_pairs_path': 'data_new/fixed_test_pairs.json',
    
    # Output
    'output_dir': 'models/unsupervised_simcse',
}

# Models to test - from Form B proposal + MTEB leaderboard top performers
MODELS_TO_TEST = {
    # Baseline (current production baseline)
    'all-mpnet-base-v2': {
        'path': 'sentence-transformers/all-mpnet-base-v2',
        'source': 'Baseline',
        'params': '109M',
        'dims': 768,
    },
    # From Form B proposal - lightweight
    'all-MiniLM-L6-v2': {
        'path': 'sentence-transformers/all-MiniLM-L6-v2',
        'source': 'Form B',
        'params': '22M',
        'dims': 384,
    },
    # MTEB Top Performers (Open Source)
    'bge-m3': {
        'path': 'BAAI/bge-m3',
        'source': 'MTEB (63.0)',
        'params': '568M',
        'dims': 1024,
    },
    'NV-Embed-v2': {
        'path': 'nvidia/NV-Embed-v2',
        'source': 'MTEB (69.32)',
        'params': '7.8B',
        'dims': 4096,
    },
    'e5-mistral-7b': {
        'path': 'intfloat/e5-mistral-7b-instruct',
        'source': 'MTEB (61.8)',
        'params': '7B',
        'dims': 4096,
    },
    # From Form B proposal - Qwen large model
    'gte-Qwen2-7B': {
        'path': 'Alibaba-NLP/gte-Qwen2-7B-instruct',
        'source': 'Form B / MTEB',
        'params': '7B',
        'dims': 4096,
    },
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

print(f"\nModels to test: {len(MODELS_TO_TEST)}")
for name, info in MODELS_TO_TEST.items():
    print(f"  - {name}: {info['params']} params, {info['dims']}-dim ({info['source']})")

In [10]:
# Cell 3: Load Training Data (raw tickets)
print(f"Loading data from {CONFIG['data_path']}...")

df = pd.read_csv(CONFIG['data_path'])
print(f"Total rows: {len(df)}")

# Extract text column
text_column = CONFIG['text_column']
if text_column not in df.columns:
    print(f"Warning: '{text_column}' not found, falling back to 'Description'")
    text_column = 'Description'

# Get non-empty texts
train_sentences = df[text_column].dropna().astype(str).tolist()
train_sentences = [s.strip() for s in train_sentences if len(s.strip()) > 20]

if CONFIG['max_samples']:
    train_sentences = train_sentences[:CONFIG['max_samples']]

print(f"Training sentences: {len(train_sentences)}")
print(f"\nSample texts:")
for i, text in enumerate(train_sentences[:3]):
    print(f"  [{i+1}] {text[:100]}...")

Loading data from data/incident_ticket_data_redacted.csv...
Total rows: 1000
Training sentences: 982

Sample texts:
  [1] grpt not working as expected. zmmm_po_rev is not generating correct dates as per maintained in grpt ...
  [2] etr-s1-24000073 exchange rate did not auto populate after user had input in exchange rate tab. syste...
  [3] there is no grpt maintenance for sold-to: 39012, vendor : 27857, cust ship mode: 19. for zor5 orders...


In [11]:
# Cell 4: Load Test Pairs (for evaluation)
test_pairs_path = CONFIG['test_pairs_path']

if os.path.exists(test_pairs_path):
    with open(test_pairs_path, 'r') as f:
        test_data = json.load(f)
    
    test_texts1 = test_data['texts1']
    test_texts2 = test_data['texts2']
    test_labels = test_data['labels']
    
    print(f"Test pairs loaded: {len(test_texts1)}")
    print(f"Label distribution: {np.unique(test_labels, return_counts=True)}")
else:
    print(f"Warning: Test pairs not found at {test_pairs_path}")
    print("Evaluation will be skipped.")
    test_texts1, test_texts2, test_labels = None, None, None

Test pairs loaded: 1000
Label distribution: (array([0, 1]), array([500, 500]))


In [12]:
# Cell 5: Initialize Model
print(f"Loading base model: {CONFIG['base_model']}")

model = SentenceTransformer(CONFIG['base_model'], device=str(device))

print(f"Model loaded on {device}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading base model: sentence-transformers/all-mpnet-base-v2
Model loaded on cuda
Embedding dimension: 768


In [13]:
# Cell 6: Evaluation Function
def evaluate_model(model, texts1, texts2, labels, description=""):
    """Evaluate model on test pairs."""
    if texts1 is None:
        print("No test data available")
        return {}
    
    print(f"\n{'='*50}")
    print(f"Evaluation: {description}")
    print(f"{'='*50}")
    
    # Get embeddings
    embeddings1 = model.encode(texts1, convert_to_tensor=True, show_progress_bar=True)
    embeddings2 = model.encode(texts2, convert_to_tensor=True, show_progress_bar=True)
    
    # Calculate cosine similarities
    from torch.nn.functional import cosine_similarity
    similarities = cosine_similarity(embeddings1, embeddings2).cpu().numpy()
    
    # Metrics
    spearman = spearmanr(similarities, labels)[0]
    pearson = pearsonr(similarities, labels)[0]
    
    # For ROC-AUC, need binary labels
    binary_labels = [1 if l > 0.5 else 0 for l in labels]
    roc_auc = roc_auc_score(binary_labels, similarities)
    
    results = {
        'spearman': spearman,
        'pearson': pearson,
        'roc_auc': roc_auc,
        'mean_similarity': float(np.mean(similarities)),
        'std_similarity': float(np.std(similarities))
    }
    
    print(f"Spearman correlation: {spearman:.4f}")
    print(f"Pearson correlation:  {pearson:.4f}")
    print(f"ROC-AUC:              {roc_auc:.4f}")
    print(f"Mean similarity:      {results['mean_similarity']:.4f} ± {results['std_similarity']:.4f}")
    
    return results

In [ ]:
# Cell 6b: Multi-Model Evaluation Function
import time
import gc

def evaluate_all_models(models_dict, texts1, texts2, labels, include_v4=True):
    """
    Evaluate multiple models systematically.
    Clears GPU memory between models to handle large 7B+ models.
    """
    all_results = {}
    
    for name, info in models_dict.items():
        print(f"\n{'='*60}")
        print(f"Evaluating: {name}")
        print(f"  Path: {info['path']}")
        print(f"  Params: {info['params']}, Dims: {info['dims']}")
        print(f"{'='*60}")
        
        try:
            start_time = time.time()
            
            # Load model with fp16 for large models
            if '7B' in info['params'] or '8B' in info['params']:
                print("  Loading with float16 for memory efficiency...")
                model = SentenceTransformer(
                    info['path'], 
                    device=str(device),
                    model_kwargs={'torch_dtype': torch.float16}
                )
            else:
                model = SentenceTransformer(info['path'], device=str(device))
            
            load_time = time.time() - start_time
            print(f"  Model loaded in {load_time:.1f}s")
            print(f"  Embedding dimension: {model.get_sentence_embedding_dimension()}")
            
            # Evaluate
            eval_start = time.time()
            results = evaluate_model(model, texts1, texts2, labels, name)
            eval_time = time.time() - eval_start
            
            # Add metadata
            results['model_name'] = name
            results['params'] = info['params']
            results['dims'] = info['dims']
            results['source'] = info['source']
            results['load_time'] = load_time
            results['eval_time'] = eval_time
            
            all_results[name] = results
            
            # Cleanup
            del model
            torch.cuda.empty_cache()
            gc.collect()
            print(f"  GPU memory cleared")
            
        except Exception as e:
            print(f"  FAILED: {e}")
            all_results[name] = {
                'error': str(e),
                'model_name': name,
                'params': info['params'],
                'dims': info['dims'],
                'source': info['source'],
            }
    
    # Optionally add V4 Cosine
    if include_v4:
        v4_path = 'models/real_servicenow_finetuned_mpnet_lora/real_servicenow_v2_20260104_2321'
        if os.path.exists(v4_path):
            print(f"\n{'='*60}")
            print("Evaluating: V4 Cosine (Fine-tuned Production)")
            print(f"{'='*60}")
            try:
                v4_model = SentenceTransformer(v4_path, device=str(device))
                results = evaluate_model(v4_model, texts1, texts2, labels, "V4 Cosine")
                results['model_name'] = 'V4 Cosine'
                results['params'] = '109M (fine-tuned)'
                results['dims'] = 768
                results['source'] = 'Production'
                all_results['V4 Cosine'] = results
                del v4_model
                torch.cuda.empty_cache()
            except Exception as e:
                print(f"  FAILED: {e}")
    
    return all_results

print("Multi-model evaluation function defined.")

In [14]:
# Cell 7: Baseline Evaluation (before training)
baseline_results = evaluate_model(
    model, test_texts1, test_texts2, test_labels,
    description="Baseline (pre-trained, no fine-tuning)"
)


Evaluation: Baseline (pre-trained, no fine-tuning)


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Spearman correlation: 0.5038
Pearson correlation:  0.5243
ROC-AUC:              0.7909
Mean similarity:      0.4480 ± 0.1778


In [15]:
# Cell 8: Create SimCSE Dataset
# SimCSE: Each sentence is paired with itself
# The model's dropout creates different representations for positive pairs
# Other sentences in the batch serve as negatives

print("Creating SimCSE dataset...")
print("Method: Each sentence paired with itself (dropout creates augmentation)")

# Create training examples - each sentence paired with itself
train_examples = [
    InputExample(texts=[sent, sent])
    for sent in train_sentences
]

train_dataloader = DataLoader(
    train_examples,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    drop_last=True
)

print(f"Training examples: {len(train_examples)}")
print(f"Batches per epoch: {len(train_dataloader)}")
print(f"Effective negatives per sample: {CONFIG['batch_size'] - 1}")

Creating SimCSE dataset...
Method: Each sentence paired with itself (dropout creates augmentation)
Training examples: 982
Batches per epoch: 15
Effective negatives per sample: 63


In [16]:
# Cell 9: Define SimCSE Loss
# MultipleNegativesRankingLoss treats other samples in batch as negatives
# Combined with dropout augmentation = SimCSE

print("Setting up SimCSE loss (MultipleNegativesRankingLoss)...")

train_loss = losses.MultipleNegativesRankingLoss(model)

print("SimCSE loss configured")
print("\nHow it works:")
print("  1. Same sentence passed twice through encoder")
print("  2. Dropout creates different representations (positive pair)")
print("  3. Other sentences in batch = negatives")
print("  4. Contrastive loss pulls positives together, pushes negatives apart")

Setting up SimCSE loss (MultipleNegativesRankingLoss)...
SimCSE loss configured

How it works:
  1. Same sentence passed twice through encoder
  2. Dropout creates different representations (positive pair)
  3. Other sentences in batch = negatives
  4. Contrastive loss pulls positives together, pushes negatives apart


In [17]:
# Cell 10: Train with SimCSE
print("\n" + "="*50)
print("Starting SimCSE Training")
print("="*50)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"{CONFIG['output_dir']}/simcse_{timestamp}"

# Calculate warmup steps
total_steps = len(train_dataloader) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])

print(f"Total steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

# Training
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=CONFIG['epochs'],
    warmup_steps=warmup_steps,
    optimizer_params={'lr': CONFIG['learning_rate']},
    output_path=output_path,
    show_progress_bar=True,
    use_amp=True  # Mixed precision for speed
)

print(f"\nTraining complete! Model saved to {output_path}")


Starting SimCSE Training
Total steps: 15
Warmup steps: 1


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss



Training complete! Model saved to models/unsupervised_simcse/simcse_20260205_163835


In [18]:
# Cell 11: Post-Training Evaluation
simcse_results = evaluate_model(
    model, test_texts1, test_texts2, test_labels,
    description="SimCSE Fine-tuned"
)


Evaluation: SimCSE Fine-tuned


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Spearman correlation: 0.4868
Pearson correlation:  0.4994
ROC-AUC:              0.7810
Mean similarity:      0.2438 ± 0.2113


In [19]:
# Cell 12: Compare with V4 Cosine (Current Production)
print("\n" + "="*50)
print("Loading V4 Cosine model for comparison...")
print("="*50)

v4_model_path = 'models/real_servicenow_finetuned_mpnet_lora/real_servicenow_v2_20260104_2321'

if os.path.exists(v4_model_path):
    v4_model = SentenceTransformer(v4_model_path, device=str(device))
    v4_results = evaluate_model(
        v4_model, test_texts1, test_texts2, test_labels,
        description="V4 Cosine (Curriculum Contrastive)"
    )
else:
    print(f"V4 model not found at {v4_model_path}")
    v4_results = None


Loading V4 Cosine model for comparison...

Evaluation: V4 Cosine (Curriculum Contrastive)


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Spearman correlation: 0.5007
Pearson correlation:  0.5268
ROC-AUC:              0.7891
Mean similarity:      0.4830 ± 0.2138


In [20]:
# Cell 13: Results Summary
print("\n" + "="*60)
print("RESULTS SUMMARY: Unsupervised vs Supervised")
print("="*60)

results_table = {
    'Model': [],
    'Training': [],
    'Spearman': [],
    'ROC-AUC': [],
}

# Baseline
if baseline_results:
    results_table['Model'].append('all-mpnet-base-v2')
    results_table['Training'].append('None (zero-shot)')
    results_table['Spearman'].append(f"{baseline_results['spearman']:.4f}")
    results_table['ROC-AUC'].append(f"{baseline_results['roc_auc']:.4f}")

# SimCSE
if simcse_results:
    results_table['Model'].append('SimCSE')
    results_table['Training'].append('Unsupervised (dropout)')
    results_table['Spearman'].append(f"{simcse_results['spearman']:.4f}")
    results_table['ROC-AUC'].append(f"{simcse_results['roc_auc']:.4f}")

# V4 Cosine
if v4_results:
    results_table['Model'].append('V4 Cosine')
    results_table['Training'].append('Contrastive (curriculum)')
    results_table['Spearman'].append(f"{v4_results['spearman']:.4f}")
    results_table['ROC-AUC'].append(f"{v4_results['roc_auc']:.4f}")

results_df = pd.DataFrame(results_table)
print(results_df.to_string(index=False))

print("\n" + "-"*60)
print("INTERPRETATION:")
print("-"*60)
print("""
- If SimCSE ≈ V4 Cosine: Your labeled pairs aren't adding much value
- If SimCSE < V4 Cosine: Curriculum contrastive learning is worth the effort
- If SimCSE > Baseline: Domain adaptation helps even without labels
""")


RESULTS SUMMARY: Unsupervised vs Supervised
            Model                 Training Spearman ROC-AUC
all-mpnet-base-v2         None (zero-shot)   0.5038  0.7909
           SimCSE   Unsupervised (dropout)   0.4868  0.7810
        V4 Cosine Contrastive (curriculum)   0.5007  0.7891

------------------------------------------------------------
INTERPRETATION:
------------------------------------------------------------

- If SimCSE ≈ V4 Cosine: Your labeled pairs aren't adding much value
- If SimCSE < V4 Cosine: Curriculum contrastive learning is worth the effort
- If SimCSE > Baseline: Domain adaptation helps even without labels



In [21]:
# Cell 14: Save Comparison Results
comparison_results = {
    'timestamp': timestamp,
    'config': CONFIG,
    'baseline': baseline_results,
    'simcse': simcse_results,
    'v4_cosine': v4_results,
    'training_sentences': len(train_sentences),
    'test_pairs': len(test_texts1) if test_texts1 else 0
}

results_path = f"{CONFIG['output_dir']}/comparison_results_{timestamp}.json"
os.makedirs(CONFIG['output_dir'], exist_ok=True)

with open(results_path, 'w') as f:
    json.dump(comparison_results, f, indent=2)

print(f"Results saved to {results_path}")

Results saved to models/unsupervised_simcse/comparison_results_20260205_163835.json


## Multi-Model Comparison

This section evaluates multiple embedding models from:
1. **Form B Project Proposal**: all-MiniLM-L6-v2, gte-Qwen2-7B
2. **MTEB Leaderboard Top Performers**: BGE-M3, NV-Embed-v2, E5-Mistral-7B

Reference: [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard)

In [ ]:
# Cell 15: Run Multi-Model Evaluation
print("="*70)
print("MULTI-MODEL COMPARISON: Form B Proposal + MTEB Top Performers")
print("="*70)
print(f"\nTesting {len(MODELS_TO_TEST)} models on {len(test_texts1)} test pairs...")
print("Note: Large 7B+ models use float16 to fit in GPU memory.")
print("      Each model is unloaded after evaluation to free memory.\n")

# Run evaluation on all models
all_model_results = evaluate_all_models(
    MODELS_TO_TEST, 
    test_texts1, 
    test_texts2, 
    test_labels,
    include_v4=True  # Include V4 Cosine production model
)

print("\n" + "="*70)
print("Multi-model evaluation complete!")
print("="*70)

In [ ]:
# Cell 16: Multi-Model Results Comparison Table
import matplotlib.pyplot as plt

print("\n" + "="*80)
print("COMPREHENSIVE MODEL COMPARISON RESULTS")
print("="*80)

# Build comparison DataFrame
comparison_data = []
for name, results in all_model_results.items():
    if 'error' in results:
        comparison_data.append({
            'Model': name,
            'Source': results.get('source', 'N/A'),
            'Params': results.get('params', 'N/A'),
            'Dims': results.get('dims', 'N/A'),
            'Spearman': 'FAILED',
            'ROC-AUC': 'FAILED',
            'Status': f"Error: {results['error'][:50]}..."
        })
    else:
        comparison_data.append({
            'Model': name,
            'Source': results.get('source', 'N/A'),
            'Params': results.get('params', 'N/A'),
            'Dims': results.get('dims', 'N/A'),
            'Spearman': f"{results['spearman']:.4f}",
            'ROC-AUC': f"{results['roc_auc']:.4f}",
            'Pearson': f"{results['pearson']:.4f}",
            'Mean Sim': f"{results['mean_similarity']:.3f}",
            'Load Time': f"{results.get('load_time', 0):.1f}s",
            'Eval Time': f"{results.get('eval_time', 0):.1f}s",
        })

comparison_df = pd.DataFrame(comparison_data)

# Sort by Spearman (best first)
comparison_df_sorted = comparison_df.copy()
comparison_df_sorted['Spearman_num'] = pd.to_numeric(comparison_df_sorted['Spearman'], errors='coerce')
comparison_df_sorted = comparison_df_sorted.sort_values('Spearman_num', ascending=False)
comparison_df_sorted = comparison_df_sorted.drop('Spearman_num', axis=1)

print("\nResults sorted by Spearman correlation (best first):")
print(comparison_df_sorted.to_string(index=False))

# Identify best model
valid_results = {k: v for k, v in all_model_results.items() if 'error' not in v}
if valid_results:
    best_model = max(valid_results.items(), key=lambda x: x[1]['spearman'])
    print(f"\n🏆 BEST MODEL: {best_model[0]}")
    print(f"   Spearman: {best_model[1]['spearman']:.4f}")
    print(f"   ROC-AUC:  {best_model[1]['roc_auc']:.4f}")

In [ ]:
# Cell 17: Visualization - Model Performance Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Prepare data for plotting
valid_models = {k: v for k, v in all_model_results.items() if 'error' not in v}
model_names = list(valid_models.keys())
spearman_scores = [valid_models[m]['spearman'] for m in model_names]
roc_auc_scores = [valid_models[m]['roc_auc'] for m in model_names]

# Sort by Spearman
sorted_indices = np.argsort(spearman_scores)[::-1]
model_names_sorted = [model_names[i] for i in sorted_indices]
spearman_sorted = [spearman_scores[i] for i in sorted_indices]
roc_auc_sorted = [roc_auc_scores[i] for i in sorted_indices]

# Color by source
colors = []
for m in model_names_sorted:
    source = valid_models[m].get('source', '')
    if 'MTEB' in source:
        colors.append('#2ecc71')  # Green for MTEB
    elif 'Form B' in source:
        colors.append('#3498db')  # Blue for Form B
    elif 'Production' in source:
        colors.append('#e74c3c')  # Red for Production
    else:
        colors.append('#9b59b6')  # Purple for baseline

# Plot 1: Spearman Correlation
ax1 = axes[0]
bars1 = ax1.barh(model_names_sorted, spearman_sorted, color=colors)
ax1.set_xlabel('Spearman Correlation', fontsize=12)
ax1.set_title('Spearman Correlation by Model', fontsize=14)
ax1.set_xlim(0, max(spearman_sorted) * 1.1)
for i, (bar, score) in enumerate(zip(bars1, spearman_sorted)):
    ax1.text(score + 0.01, bar.get_y() + bar.get_height()/2, 
             f'{score:.4f}', va='center', fontsize=10)

# Plot 2: ROC-AUC
ax2 = axes[1]
bars2 = ax2.barh(model_names_sorted, roc_auc_sorted, color=colors)
ax2.set_xlabel('ROC-AUC', fontsize=12)
ax2.set_title('ROC-AUC by Model', fontsize=14)
ax2.set_xlim(0, 1.0)
for i, (bar, score) in enumerate(zip(bars2, roc_auc_sorted)):
    ax2.text(score + 0.01, bar.get_y() + bar.get_height()/2, 
             f'{score:.4f}', va='center', fontsize=10)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='MTEB Leaderboard'),
    Patch(facecolor='#3498db', label='Form B Proposal'),
    Patch(facecolor='#e74c3c', label='Production (V4)'),
    Patch(facecolor='#9b59b6', label='Baseline'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02))

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/model_comparison_{timestamp}.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nVisualization saved to {CONFIG['output_dir']}/model_comparison_{timestamp}.png")

In [ ]:
# Cell 18: Save Multi-Model Comparison Results
multi_model_results = {
    'timestamp': timestamp,
    'test_pairs_count': len(test_texts1),
    'models_tested': list(MODELS_TO_TEST.keys()) + ['V4 Cosine'],
    'results': {},
}

# Convert results for JSON serialization
for name, results in all_model_results.items():
    multi_model_results['results'][name] = {
        k: float(v) if isinstance(v, (np.floating, np.integer)) else v
        for k, v in results.items()
    }

# Save to file
multi_results_path = f"{CONFIG['output_dir']}/multi_model_comparison_{timestamp}.json"
with open(multi_results_path, 'w') as f:
    json.dump(multi_model_results, f, indent=2)

print(f"Multi-model results saved to: {multi_results_path}")
print(f"\nModels evaluated: {len(all_model_results)}")
print(f"Successful: {len([r for r in all_model_results.values() if 'error' not in r])}")
print(f"Failed: {len([r for r in all_model_results.values() if 'error' in r])}")

## Summary & Conclusions

### Models Evaluated

| Category | Models | Purpose |
|----------|--------|---------|
| **Baseline** | all-mpnet-base-v2 | Pre-trained, no fine-tuning |
| **Form B Proposal** | all-MiniLM-L6-v2, gte-Qwen2-7B | Project-specified models |
| **MTEB Leaderboard** | BGE-M3, NV-Embed-v2, E5-Mistral-7B | Top open-source performers |
| **Production** | V4 Cosine | Curriculum contrastive fine-tuned |
| **Unsupervised** | SimCSE | Dropout-based domain adaptation |

### Key Findings

1. **Larger models ≠ Better performance**: MTEB leaderboard rankings are based on general benchmarks. Domain-specific tasks (IT tickets) may favor different models.

2. **Fine-tuning value**: Compare V4 Cosine vs baseline to see curriculum learning impact:
   - If V4 ≈ Baseline: Fine-tuning didn't help much
   - If V4 > Baseline: Curriculum contrastive is adding value

3. **Unsupervised adaptation**: SimCSE provides a zero-label baseline:
   - If SimCSE > Baseline: Domain adaptation helps even without labels
   - If SimCSE < Baseline: Dropout augmentation may not suit this domain

### Recommendations

Based on results, consider:

- **Best Spearman**: Use for production semantic search
- **Best size/performance ratio**: For latency-sensitive deployments
- **V4 Cosine**: Continue using if it outperforms larger models (it's fine-tuned for YOUR data)

### Next Steps

1. **If an MTEB model wins**: Fine-tune it with curriculum contrastive learning
2. **If V4 Cosine wins**: Current production choice is validated
3. **For deployment**: Consider model size vs performance trade-off for latency requirements